# Лемматизация отзывов с pymorphy3

Локальный блокнот к урокам главы 13 «Текст как данные: очистка и токены» и «Леммы и словарь исключений». В браузере `pymorphy3` не работает, поэтому здесь — то, что на сайте показано готовой таблицей `review_lemmas.csv`:

1. как устроен разбор слова: варианты, теги, уверенность;
2. что лемматизатор делает со словами, которых нет в словаре;
3. лемматизация 2400 отзывов с кэшем — и почему без кэша медленно;
4. свой словарь исключений для слов меню;
5. лемматизатор внутри конвейера sklearn: помогает ли он модели тональности.

**Как запустить.** Блокнот лежит в `site/notebooks/extra/`, данные берёт из `site/data/`. Нужна одна дополнительная библиотека (около 10 МБ со словарём):

```sh
uv add pymorphy3          # в проекте с uv
# или прямо из блокнота: %pip install pymorphy3
```

Затем *Kernel → Restart Kernel and Run All Cells*. Весь блокнот выполняется меньше чем за минуту.

In [1]:
# Импорты, версии и общие настройки
import re
import sys
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
import pymorphy3
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42
DATA = Path('../../data')      # блокнот в site/notebooks/extra, данные в site/data

print('Python', sys.version.split()[0])
for lib in (pd, sklearn, pymorphy3):
    print(f'{lib.__name__:<11}{lib.__version__}')

Python 3.13.16
pandas     3.0.5
sklearn    1.9.1
pymorphy3  2.0.6


## 1. Разбор слова

`MorphAnalyzer` загружает словарь один раз (около секунды) — создавайте его один раз на весь блокнот. `parse(слово)` возвращает **список** разборов, отсортированный по уверенности: у русского слова часто несколько прочтений.

In [2]:
morph = pymorphy3.MorphAnalyzer()

for p in morph.parse('стали'):
    print(f'{p.normal_form:<8} {str(p.tag):<38} score={p.score:.3f}')

стать    VERB,perf,intr plur,past,indc          score=0.975
сталь    NOUN,inan,femn sing,gent               score=0.011
сталь    NOUN,inan,femn plur,nomn               score=0.005
сталь    NOUN,inan,femn sing,datv               score=0.003
сталь    NOUN,inan,femn sing,loct               score=0.003
сталь    NOUN,inan,femn plur,accs               score=0.003


Что здесь видно:

- `normal_form` — лемма: «стали» — это либо глагол «стать», либо существительное «сталь» в разных падежах;
- `tag` — грамматика: часть речи (`VERB`, `NOUN`), число, падеж, время;
- `score` — оценка вероятности разбора по корпусу. Первый разбор («стать», почти 0.98) — почти всегда верный для отзывов, поэтому дальше берём `parse(w)[0]`.

Контекст лемматизатор не учитывает: в «ложки из стали» он всё равно выберет «стать». Для отзывов о кофейне это не страшно, для текстов о металлургии — уже да.

In [3]:
for w in ['привезли', 'лучший', 'сырниками', 'вкуснейший', 'курьером']:
    p = morph.parse(w)[0]
    print(f'{w:<11} → {p.normal_form:<9} {p.tag.POS:<5} known={morph.word_is_known(w)}')

привезли    → привезти  VERB  known=True
лучший      → хороший   ADJF  known=True
сырниками   → сырник    NOUN  known=True
вкуснейший  → вкусный   ADJF  known=True
курьером    → курьер    NOUN  known=True


### Вывод по разделу 1

Для слов из словаря лемматизатор делает то, что стемминг не умеет: «лучший» → «хороший», «привезли» → «привезти». Разборов у слова может быть несколько; первый — самый вероятный, но выбирается без учёта соседних слов.

## 2. Слова, которых нет в словаре

Если слова нет в словаре, `pymorphy3` не сдаётся, а **угадывает** разбор по окончанию похожих слов. `word_is_known` говорит, был ли разбор из словаря.

In [4]:
menu = ['капучино', 'латте', 'бариста', 'раф', 'флэт', 'круассаны', 'чизкейк', 'матча']
for w in menu:
    p = morph.parse(w)[0]
    print(f'{w:<10} → {p.normal_form:<10} {str(p.tag.POS):<5} known={morph.word_is_known(w)} score={p.score:.2f}')

капучино   → капучиный  ADJS  known=False score=0.40
латте      → латта      NOUN  known=False score=0.35
бариста    → барист     NOUN  known=False score=0.21
раф        → раф        NOUN  known=True score=0.33
флэт       → флэта      NOUN  known=False score=0.50
круассаны  → круассан   NOUN  known=False score=0.17
чизкейк    → чизкейк    NOUN  known=True score=0.50
матча      → матч       NOUN  known=True score=1.00


### Вывод по разделу 2

Названия напитков и профессий кофейни словарь знает плохо: «капучино» становится прилагательным «капучиный», «латте» — «латта», «бариста» — «барист». Обратная ловушка — «матча»: слово в словаре есть, но это форма слова «матч», и лемматизатор уверенно (score 1.00) превращает чай в футбол. `known=True` не гарантирует, что смысл тот. Сами по себе догадки безвредны, пока у всех форм слова одна и та же неверная лемма. Плохо, когда в отчёт о напитках попадает «капучиный», — поэтому частые незнакомые слова просматривают глазами (раздел 4).

## 3. Лемматизация всех отзывов и кэш

Разбор слова — дорогая операция, а разных слов в отзывах мало: 2400 отзывов — это около 19 тысяч словоупотреблений, но меньше 500 разных слов. Значит, лемму нужно считать **один раз на слово**. Проще всего — декоратор `lru_cache`: функция запоминает ответы для уже встречавшихся аргументов.

In [5]:
reviews = pd.read_csv(DATA / 'reviews_text.csv')

def tokenize(text):
    return re.findall(r'[а-яё]+', text.lower())

def lemma_nocache(word):
    return morph.parse(word)[0].normal_form

@lru_cache(maxsize=None)
def lemma(word):
    return morph.parse(word)[0].normal_form

tokens = [w for t in reviews['text'] for w in tokenize(t)]
print('словоупотреблений:', len(tokens), '| разных слов:', len(set(tokens)))

start = time.perf_counter()
plain = [lemma_nocache(w) for w in tokens]
t_plain = time.perf_counter() - start

start = time.perf_counter()
cached = [lemma(w) for w in tokens]
t_cached = time.perf_counter() - start

print(f'без кэша: {t_plain:.2f} с | с кэшем: {t_cached:.2f} с | совпадают: {plain == cached}')
print(lemma.cache_info())

словоупотреблений:

 19257 | разных слов: 491


без кэша: 12.02 с | с кэшем: 0.69 с | совпадают: True
CacheInfo(hits=18766, misses=491, maxsize=None, currsize=491)


`cache_info()` показывает: промахов (`misses`) столько же, сколько разных слов, остальные обращения — попадания в кэш. На миллионе отзывов разница — минуты против секунд.

Соберём ту же таблицу «слово → лемма», что лежит на сайте, и сверим с ней.

In [6]:
counts = Counter(tokens)
rows = []
for w, n in counts.items():
    p = morph.parse(w)[0]
    rows.append({'word': w, 'lemma': p.normal_form, 'pos': p.tag.POS or '',
                 'known': int(morph.word_is_known(w)), 'count': n})
table = pd.DataFrame(rows).sort_values(['count', 'word'], ascending=[False, True], ignore_index=True)

site = pd.read_csv(DATA / 'review_lemmas.csv')
same = table.set_index('word')['lemma'].eq(site.set_index('word')['lemma']).mean()
print('доля совпадений с review_lemmas.csv:', same)
print('слов:', table['word'].nunique(), '| лемм:', table['lemma'].nunique(),
      '| основ по 5 букв:', len({w[:5] for w in counts}))
table.head(10)

доля совпадений с review_lemmas.csv: 1.0
слов: 491 | лемм: 450 | основ по 5 букв: 377


,word,lemma,pos,known,count
0,не,не,PRCL,1,561
1,кофе,кофе,NOUN,1,412
2,с,с,PREP,1,405
3,но,но,CONJ,1,354
4,приду,прийти,VERB,1,311
5,и,и,CONJ,1,306
6,у,у,PREP,1,304
7,советую,советовать,VERB,1,287
8,через,через,PREP,1,287
9,в,в,PREP,1,282


### Вывод по разделу 3

Лемматизация сводит 491 слово к 450 леммам — скромно, но без ложных склеек. Кэш обязателен: лемма считается один раз на разное слово. Таблица совпадает с `review_lemmas.csv` (если версия словаря у вас другая, возможны расхождения в единичных словах).

## 4. Свой словарь исключений

Смотрим частые незнакомые слова и правим их леммы. Правка — на уровне **леммы**: тогда все формы слова, которые лемматизатор свёл к неверной лемме, исправятся разом.

In [7]:
suspects = table[(table['known'] == 0) & (table['count'] >= 30)]
print(suspects[['word', 'lemma', 'count']].to_string(index=False))

FIXES = {'капучиный': 'капучино', 'латта': 'латте', 'барист': 'бариста', 'флэта': 'флэт'}

@lru_cache(maxsize=None)
def lemma_fixed(word):
    l = lemma(word)
    return FIXES.get(l, l)

after = Counter(lemma_fixed(w) for w in tokens)
print({d: after[d] for d in ['кофе', 'капучино', 'латте', 'раф', 'флэт', 'бариста']})

       word       lemma  count
   капучино   капучиный    181
    бариста      барист    145
      латте       латта    108
       флэт       флэта     72
  круассаны    круассан     40
   круассан    круассан     35
начислились начислиться     34
{'кофе': 412, 'капучино': 181, 'латте': 108, 'раф': 111, 'флэт': 72, 'бариста': 146}


### Вывод по разделу 4

Семь частых незнакомых слов — это меню, профессия и глагол «начислились» (про бонусы), лемму которого pymorphy3 угадал верно. Четыре правки в словаре исключений возвращают в отчёт «капучино», «латте», «бариста» и «флэт». Словарь исключений хранят рядом с кодом и пополняют при появлении новых позиций меню.

## 5. Лемматизатор в конвейере sklearn

`TfidfVectorizer` принимает свою функцию разбиения на токены: `tokenizer=...`. Передадим ему три варианта — исходные слова, леммы с исключениями и обрезание до 5 букв — и сравним F1 модели тональности (недоволен = оценка не выше 3) на 5-кратной кросс-валидации.

In [8]:
y = (reviews['rating'] <= 3).astype(int)

variants = {
    'слова':  tokenize,
    'леммы':  lambda t: [lemma_fixed(w) for w in tokenize(t)],
    'основы': lambda t: [w[:5] for w in tokenize(t)],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
result = {}
for name, tok in variants.items():
    model = make_pipeline(TfidfVectorizer(tokenizer=tok, token_pattern=None, lowercase=False),
                          LogisticRegression(max_iter=1000))
    scores = cross_val_score(model, reviews['text'], y, cv=cv, scoring='f1')
    result[name] = (scores.mean(), scores.std())
    print(f'{name:<7} F1 = {scores.mean():.3f} ± {scores.std():.3f}')

слова   F1 = 0.940 ± 0.015


леммы   F1 = 0.940 ± 0.015


основы  F1 = 0.940 ± 0.015


Что здесь происходит:

- `tokenizer=tok` — векторайзер разбивает текст нашей функцией; `token_pattern=None` отключает встроенное регулярное выражение (иначе sklearn предупредит, что оно не используется);
- `lowercase=False` — регистр уже приводит `tokenize`;
- `StratifiedKFold(..., random_state=RANDOM_STATE)` — одно и то же разбиение для всех трёх вариантов, чтобы сравнение было парным.

### Вывод по разделу 5

На коротких шаблонных отзывах все три варианта дают одинаковый F1 (до третьего знака): формы слов здесь почти не мешают модели. Это нормальный результат: лемматизация сильнее помогает на маленьких выборках и длинных разнообразных текстах, где каждая форма слова встречается редко, — и в отчётах, где нужно считать упоминания («бариста» во всех формах). Решение «нужна ли лемматизация» принимают по кросс-валидации, а не по привычке.

## Попробуйте сами

1. Добавьте в `variants` вариант «леммы без исключений» (`lemma` вместо `lemma_fixed`). Изменился ли F1? А частоты напитков?
2. Возьмите `reviews_new.csv` (новые отзывы с карт): сколько там незнакомых словарю слов встречается чаще 30 раз? Пополните `FIXES`.
3. Уберите стоп-слова **после** лемматизации, со списком в начальной форме (`быть`, `этот`, `весь`, …). Сравните размер словаря до и после.